<p align="center">
  <img src="https://raw.githubusercontent.com/CSSB-SNU/Thal-Kak/main/assets/thalkak_w_plang.png" height="200" style="vertical-align: middle;">
</p>

End-to-end protein structure prediction: **MSA** (ColabFold) → **structure** (Boltz-2 / Chai-1 / Protenix / ESMFold2) → **OpenMM relaxation**, with per-model confidence-based top-5 selection.

**How to run:** set your inputs in step 1, then `Runtime` → `Run all`.

> **Requirements**
> - A **GPU** runtime: `Runtime` → `Change runtime type` → **GPU** (T4 is fine).
> - Protein targets only (RNA needs a large local database built offline).
> - Environment install is fast (**~3 min** via pixi); the first run also downloads model weights (a few minutes).
> - **Memory:** the standard runtime has ~12 GB RAM. Chai-1, Protenix, and ESMFold2 load a large ESM model, so multi-chain or otherwise large inputs can exhaust RAM and crash the kernel. For those, switch to a **High-RAM runtime** (`Runtime` → `Change runtime type`) or reduce the workload (fewer seeds, smaller inputs). Colab does not permit adding swap.

In [ ]:
#@title 1. Input sequence & options
import os

jobname = "insulin" #@param {type:"string"}
#@markdown - Protein sequence. For a **complex**, separate chains with `:`.
query_sequence = "GIVEQCCTSICSLYQLENYCN:FVNQHLCGSHLVEALYLVCGERGFFYTPKT" #@param {type:"string"}
#@markdown - Stoichiometry, e.g. `A1` (monomer), `A2` (homodimer), `A1B1`
#@markdown   (heterodimer). Leave as `UNK` to use one copy of each chain.
stoichiometry = "A1B1" #@param {type:"string"}
#@markdown ---
#@markdown ### Model & run options
structure_model = "boltz2" #@param ["boltz2", "chai1", "protenix_v1", "protenix_v2", "esmfold2"]
relax_method = "openmm" #@param ["openmm", "none"]
num_seeds = 5 #@param {type:"integer"}
#@markdown - Confidence metric that ranks the top-5 models.
top5_metric = "ranking_score" #@param ["ranking_score", "ptm", "iptm", "plddt"]
#@markdown ---
#@markdown ### Google Drive cache (optional)
#@markdown Cache the selected model's large checkpoint file(s) on your Drive
#@markdown so later sessions skip that (large) download. The first successful
#@markdown run fills it; later runs read the checkpoint straight from Drive.
#@markdown Small files still download each run. Uncheck to skip caching.
use_drive_cache = True #@param {type:"boolean"}
drive_cache_dir = "/content/drive/MyDrive/thalkak_cache" #@param {type:"string"}

# Build a full-mode input yaml (Method + Entity) for `thalkak full --input`.
import re
seqs = [s.strip().upper().replace(" ", "")
        for s in query_sequence.split(":") if s.strip()]
assert seqs, "query_sequence is empty"
workdir = f"/content/{jobname}"
os.makedirs(workdir, exist_ok=True)
# Map stoichiometry (e.g. "A2B1") to a per-chain copy count; "UNK" -> 1 each.
if stoichiometry.strip().upper() == "UNK":
    copies = [1] * len(seqs)
else:
    _counts = dict(re.findall(r"([A-Za-z])(\d+)", stoichiometry))
    copies = [int(_counts.get(chr(65 + i), 1)) for i in range(len(seqs))]
input_yaml = f"{workdir}/input.yaml"
with open(input_yaml, "w") as f:
    f.write("Method:\n")
    f.write(f"  jobname: {jobname}\n")
    f.write("  msa: colab\n")
    f.write(f"  structure: {structure_model}\n")
    f.write(f"  relax: {relax_method}\n")
    f.write(f"  top5_metric: {top5_metric}\n")
    f.write(f"  n_seed: {int(num_seeds)}\n")
    f.write("Entity:\n")
    for i, s in enumerate(seqs):
        f.write("  - type: protein\n")
        f.write(f"    seq: {s}\n")
        f.write(f"    copy: {copies[i]}\n")

print(f"Wrote input yaml ({len(seqs)} chain(s)) to {input_yaml}")
print(f"model={structure_model}  relax={relax_method}  "
      f"seeds={num_seeds}  stoi={stoichiometry}")

# Weight cache for the SELECTED model only. Everything downloads to local
# disk; only large checkpoint files are mirrored to Drive (as themselves,
# not a tar). Small files -- notably boltz's ~45k CCD mol pickles -- always
# download locally, so they never clutter Drive or hit its FUSE small-file
# limits. A cached checkpoint is symlinked back in and read straight from Drive.
LOCAL_CACHE = "/content/thalkak_cache"
BIG_FILE_BYTES = 50 * 1024 * 1024   # a file this big or larger is a checkpoint
_MODEL_CACHE = {"boltz2": ("BOLTZ_CACHE", "boltz"),
                "chai1": ("CHAI_DOWNLOADS_DIR", "chai"),
                "protenix_v1": ("PROTENIX_CHECKPOINT_DIR", "protenix"),
                "protenix_v2": ("PROTENIX_CHECKPOINT_DIR", "protenix"),
                "esmfold2": ("HF_HOME", "hf")}
env_var, sub = _MODEL_CACHE[structure_model]
local_sub = os.path.join(LOCAL_CACHE, sub)
os.makedirs(local_sub, exist_ok=True)
os.environ[env_var] = local_sub          # all weights download to local disk
drive_big = None   # Drive dir holding this model's large checkpoints
if use_drive_cache:
    from google.colab import drive
    drive.mount("/content/drive")
    drive_big = os.path.join(drive_cache_dir, sub + "_ckpt")
    os.makedirs(drive_big, exist_ok=True)
    # Symlink any cached checkpoint into the local cache so the backend sees
    # it as present (skips that download) and reads it directly from Drive.
    linked = 0
    for _root, _dirs, _files in os.walk(drive_big):
        for _fn in _files:
            _src = os.path.join(_root, _fn)
            _dst = os.path.join(local_sub, os.path.relpath(_src, drive_big))
            os.makedirs(os.path.dirname(_dst), exist_ok=True)
            if not os.path.lexists(_dst):
                os.symlink(_src, _dst); linked += 1
    print(f"{structure_model} weights: {linked} checkpoint file(s) linked from "
          f"Drive; any remaining files download locally.")
else:
    print("Drive cache off; weights download for this session only.")

In [ ]:
#@title 2. Clone the Thal-Kak repository
import os, subprocess

REPO = "CSSB-SNU/Thal-Kak"
REPO_DIR = "/content/Thal-Kak"

if not os.path.isdir(REPO_DIR):
    rc = subprocess.run(
        ["git", "clone", "--depth", "1",
         f"https://github.com/{REPO}.git", REPO_DIR]
    ).returncode
    if rc != 0:
        raise RuntimeError("git clone failed.")
    print("Cloned", REPO)
else:
    print("Repo already present, skipping clone.")

In [ ]:
#@title 3. Install dependencies (pixi) — ~3 min
#@markdown Installs the [pixi](https://pixi.sh) package manager, resolves the
#@markdown locked `thalkak` environment from `pixi.lock` (no dependency solve),
#@markdown and applies the ColabFold templates patch. Re-running is cheap.
import os, shlex

script = r"""
set -e
export PATH="$HOME/.pixi/bin:$PATH"
if ! command -v pixi >/dev/null 2>&1; then
  echo "== Installing pixi =="
  curl -fsSL https://pixi.sh/install.sh | bash
  export PATH="$HOME/.pixi/bin:$PATH"
fi
cd /content/Thal-Kak
echo "== Creating the thalkak env from pixi.lock =="
pixi install
echo "== Post-install (ColabFold templates patch) =="
pixi run postinstall
echo "== Install step complete =="
"""

rc = os.system("bash -c " + shlex.quote(script))
if rc != 0:
    raise RuntimeError("Installation failed — see the log above.")

In [ ]:
#@title 4. Run Thal-Kak (MSA → structure → relax)
import subprocess, shlex

cmd = (
    'export PATH="$HOME/.pixi/bin:$PATH" && '
    "cd /content/Thal-Kak && "
    # run inside the pixi env; -u so progress prints stream live in Colab
    "PYTHONUNBUFFERED=1 pixi run python -u thalkak.py full "
    f"--input {shlex.quote(input_yaml)}"
)
# Stream stdout/stderr line by line so progress shows live in the cell.
proc = subprocess.Popen(["bash", "-c", cmd], stdout=subprocess.PIPE,
                        stderr=subprocess.STDOUT, text=True, bufsize=1)
for line in proc.stdout:
    print(line, end="", flush=True)
if proc.wait() != 0:
    raise RuntimeError("Thal-Kak run failed — see the log above.")
print("Done. Outputs under:", workdir)

# Success: mirror any newly-downloaded large checkpoints to Drive. These are
# single big files (Drive FUSE handles them fine); the many small mol pickles
# are never copied. .part -> replace so an interrupted copy never looks done.
import os, shutil
if globals().get("drive_big"):
    local_sub = os.path.join(LOCAL_CACHE, sub)
    pushed = 0
    for _root, _dirs, _files in os.walk(local_sub):
        for _fn in _files:
            _src = os.path.join(_root, _fn)
            if os.path.islink(_src):
                continue                       # already backed by Drive
            try:
                if os.path.getsize(_src) < BIG_FILE_BYTES:
                    continue
            except OSError:
                continue
            _rel = os.path.relpath(_src, local_sub)
            _dst = os.path.join(drive_big, _rel)
            if os.path.exists(_dst):
                continue
            os.makedirs(os.path.dirname(_dst), exist_ok=True)
            try:
                print(f"caching checkpoint {_rel} to Drive ...", flush=True)
                shutil.copyfile(_src, _dst + ".part")   # data only (FUSE-safe)
                os.replace(_dst + ".part", _dst)
                pushed += 1
            except Exception as e:
                print(f"  (skipped {_rel}: {e})", flush=True)
    if pushed:
        print(f"cached {pushed} checkpoint file(s) to Drive.")

In [ ]:
#@title 5. View a predicted structure (per model)
#@markdown Use the dropdowns to switch model / coloring; the view updates live.
import os, glob
import ipywidgets as W
from IPython.display import display
import matplotlib.pyplot as plt, matplotlib as mpl
from matplotlib.colors import Normalize, LinearSegmentedColormap
try:
    import py3Dmol
except ImportError:
    os.system("pip -q install py3Dmol"); import py3Dmol

top5_dirs = glob.glob(f"{workdir}/top5/*_results_*")
assert top5_dirs, "No top5 dir found - did the run finish?"
top5_dir = max(top5_dirs, key=os.path.getmtime)  # newest run (stoi/params may differ)
relaxed_dirs = sorted(glob.glob(f"{top5_dir}/relaxed/*"))
relaxed_dir = relaxed_dirs[0] if relaxed_dirs else None
model_ids = sorted({int(os.path.basename(p).split("_")[1].split(".")[0])
                    for p in glob.glob(f"{top5_dir}/model_*.pdb")}) or [1]

def structure_path(n):
    if relaxed_dir:
        hits = glob.glob(f"{relaxed_dir}/model_{n}_*.pdb")
        if hits: return hits[0]
    p = f"{top5_dir}/model_{n}.pdb"
    return p if os.path.exists(p) else None

def style(color):
    # relaxed/unrelaxed PDBs carry pLDDT in the B-factor column
    if color == "pLDDT":
        return {"cartoon": {"colorscheme": {"prop": "b", "gradient": "rwb",
                                            "min": 40, "max": 100}}}
    return {"cartoon": {"color": "spectrum"}}

model_dd = W.Dropdown(options=model_ids, value=model_ids[0], description="Model:")
color_dd = W.Dropdown(options=["pLDDT", "N to C (rainbow)"], value="pLDDT",
                      description="Color:")
cbar = W.Output()
view = py3Dmol.view(width=700, height=500)

def draw_colorbar(color):
    cbar.clear_output(wait=True)
    with cbar:
        fig, ax = plt.subplots(figsize=(4.5, 0.42))
        if color == "pLDDT":
            # matplotlib "RdBu": 0->red, 1->blue (matches 3Dmol "rwb")
            cmap, norm, label = plt.get_cmap("RdBu"), Normalize(40, 100), "pLDDT"
        else:
            cmap = LinearSegmentedColormap.from_list(
                "roygb", ["red", "orange", "yellow", "green", "blue"])  # 3Dmol spectrum
            norm, label = Normalize(0, 1), "N-term  ->  C-term"
        cb = mpl.colorbar.ColorbarBase(ax, cmap=cmap, norm=norm, orientation="horizontal")
        cb.set_label(label)
        if color != "pLDDT":
            cb.set_ticks([])
        plt.show()

def refresh(_=None):
    p = structure_path(model_dd.value)
    if not p:
        return
    # mutate the existing viewer in place (no re-injection -> no blank in Colab)
    view.removeAllModels()
    view.addModel(open(p).read(), "pdb")
    view.setStyle(style(color_dd.value))
    view.zoomTo()
    view.update()
    draw_colorbar(color_dd.value)

display(W.HBox([model_dd, color_dd]))
p0 = structure_path(model_dd.value)
if p0:
    view.addModel(open(p0).read(), "pdb")
    view.setStyle(style(color_dd.value))
    view.zoomTo()
view.show()
display(cbar)
draw_colorbar(color_dd.value)

model_dd.observe(refresh, names="value")
color_dd.observe(refresh, names="value")


In [ ]:
#@title 6. Metrics (MSA depth, pAE, pLDDT, ranking, energy)
import os, glob, yaml, pandas as pd
from IPython.display import display, Image
import ipywidgets as W

top5_dirs = glob.glob(f"{workdir}/top5/*_results_*")
assert top5_dirs, "No result dirs found - did the run finish?"
top5_dir = max(top5_dirs, key=os.path.getmtime)  # newest run
struct_dir = os.path.join(workdir, "structure", os.path.basename(top5_dir))
common = os.path.join(struct_dir, "common")
relaxed_dirs = sorted(glob.glob(f"{top5_dir}/relaxed/*"))
relaxed_dir = relaxed_dirs[0] if relaxed_dirs else None

def _first_png(*pats):
    for pat in pats:
        hits = sorted(glob.glob(pat))
        if hits: return hits[0]
    return None

def show_png(caption, *pats):
    png = _first_png(*pats)
    if png: print(caption); display(Image(png))
    else: print(f"{caption}: image not found.")

def show_msa_depth():
    a3ms = [p for p in glob.glob(f"{workdir}/msa/*/*.a3m") if "_env" not in p]
    if a3ms:
        depth = sum(1 for l in open(a3ms[0]) if l.startswith(">"))
        print(f"MSA depth: {depth} sequences  ({os.path.basename(a3ms[0])})")
    show_png("Coverage", f"{workdir}/msa/*/*coverage*.png")

def show_ranking():
    csv = glob.glob(f"{common}/*results_summary.csv")
    if not csv: print("summary csv not found"); return
    print("Top-5 models are chosen by ranking_score (descending):")
    display(pd.read_csv(csv[0]).head(10))

def show_energy():
    if not relaxed_dir: print("no relaxed dir"); return
    ep = os.path.join(relaxed_dir, "energies.yaml")
    if not os.path.exists(ep): print("energies.yaml not found (relax=none?)"); return
    display(pd.DataFrame(yaml.safe_load(open(ep)) or {}).T)

TABS = [
    ("MSA depth", show_msa_depth),
    ("pAE",       lambda: show_png("pAE", f"{common}/*pae*.png")),
    ("pLDDT",     lambda: show_png("pLDDT", f"{common}/*plddt*.png")),
    ("Ranking",   show_ranking),
    ("Energy",    show_energy),
]
outs = [W.Output() for _ in TABS]
tab = W.Tab(children=outs)
for i, (t, _) in enumerate(TABS): tab.set_title(i, t)
for o, (_, fn) in zip(outs, TABS):
    with o:
        try: fn()
        except Exception as e: print("error:", e)
display(tab)


In [ ]:
#@title 7. Download results
#@markdown Zips the latest run (its structures, top-5, relaxed models,
#@markdown confidence logs) plus the MSA, and downloads it. Optionally copy
#@markdown to Google Drive.
save_to_google_drive = False #@param {type:"boolean"}
import os, glob, zipfile

# Latest run only (same selection as the viewer / metrics cells).
top5_dirs = glob.glob(f"{workdir}/top5/*_results_*")
assert top5_dirs, "No results found - did the run finish?"
top5_dir = max(top5_dirs, key=os.path.getmtime)
run_name = os.path.basename(top5_dir)
roots = [top5_dir,
         os.path.join(workdir, "structure", run_name),
         os.path.join(workdir, "msa")]

zip_path = f"/content/{jobname}.result.zip"
with zipfile.ZipFile(zip_path, "w", zipfile.ZIP_DEFLATED) as z:
    for root in roots:
        if not os.path.isdir(root): continue
        for dirpath, _, fnames in os.walk(root):
            for fn in fnames:
                full = os.path.join(dirpath, fn)
                z.write(full, os.path.relpath(full, workdir))
print("Result archive:", zip_path, f"(run: {run_name})")

if save_to_google_drive:
    import shutil
    from google.colab import drive
    drive.mount("/content/drive")
    dst = f"/content/drive/MyDrive/{jobname}.result.zip"
    shutil.copyfile(zip_path, dst)
    print("Saved to", dst)

from google.colab import files
files.download(zip_path)

## Notes

- **Models & weights.** Each backend downloads its own weights on first run (e.g. `~/.boltz` for Boltz-2, HuggingFace cache for ESMFold2). Weights are distributed under their providers' own terms.
- **Outputs.** For a job named `insulin` the results live under `/content/insulin/`: `msa/`, `structure/`, and `top5/<job>/` with `model_1..5.pdb`, `relaxed/<method>/`, and `method_log.yaml` provenance.
- **RNA/DNA.** This notebook targets proteins. RNA MSA needs a large local database (`./install_db.sh --family rna`) that is impractical to build on Colab — run RNA targets locally instead.
- **License.** Thal-Kak is Apache-2.0; see `LICENSE` and `NOTICE` in the repo.